# Generator dataset komentar sintetis

Notebook mandiri ini menambah komentar sintetis bergaya promosi judi sampai jumlah dataset mencapai target. Jalankan setelah `01_eda_cleaning.ipynb` menghasilkan `outputs/cleaning/komentar_cleaned.csv`.

Urutan: konfigurasi → baca dataset bersih → buat contoh sintetis → tinjau contoh dan distribusi → ekspor. Menjalankan sel generator hanya membuat data di memori; file baru ditulis ketika sel ekspor terakhir dijalankan. Dataset sumber dan hasil cleaning tidak ditimpa.

Dependensi: Python, pandas, dan IPython/Jupyter. Contoh tambahan bukan komentar YouTube yang dikumpulkan, dan harus dilaporkan sebagai data sintetis dalam paper.

Sumber dataset awal: [Judi Online — Kaggle](https://www.kaggle.com/datasets/yaemico/judionline), identifier `yaemico/judionline`, berdasarkan informasi anggota kelompok. Catatan sumber dan batas verifikasi tersedia di [dataset/README.md](dataset/README.md). Komentar sintetis dibuat terpisah oleh kelompok, bukan diperoleh dari Kaggle.


## 1. Konfigurasi

Ubah `TARGET_ROWS` untuk jumlah total gabungan; bukan jumlah tambahan. `SEED` mengontrol pemilihan variasi agar hasil dapat direproduksi. Hasil tidak otomatis menjadi seimbang antarkelas karena semua tambahan berlabel `1`. Pastikan arti label tersebut sesuai sumber dataset sebelum dipakai untuk eksperimen.

In [1]:
from pathlib import Path
import hashlib
import json
import random
import re
import unicodedata
import pandas as pd
from IPython.display import display

TARGET_ROWS = 10_000
SEED = 20261019
GENERATOR_VERSION = 'template_v1'

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'dataset').is_dir():
    PROJECT_DIR = PROJECT_DIR.parent
CLEANING_DIR = PROJECT_DIR / 'outputs' / 'cleaning'
cleaned_path = CLEANING_DIR / 'komentar_cleaned.csv'
SUMMARY_PATH = CLEANING_DIR / 'cleaning_summary.json'
INPUT_PATH = PROJECT_DIR / 'dataset' / 'data_komentar_dengan_prediksi - data_komentar_dengan_prediksi(2).csv'
AUGMENT_DIR = PROJECT_DIR / 'outputs' / 'augmentation'
pd.set_option('display.max_colwidth', 160)

## 2. Baca data bersih dan cek sumber

Jumlah tambahan dihitung dari file hasil cleaning yang benar-benar dibaca. Nomor sumber dipertahankan sebagai teks agar tidak berubah saat digabung dengan record sintetis yang metadata sumbernya kosong.

In [2]:
for path in [cleaned_path, SUMMARY_PATH, INPUT_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f'Tidak ditemukan: {path}. Jalankan notebook cleaning terlebih dahulu.')
cleaned = pd.read_csv(cleaned_path, dtype=str, keep_default_na=False, encoding='utf-8-sig')
required = {'komentar', 'komentar_processed', 'label', 'duplicate_count'}
assert required.issubset(cleaned.columns), f'Kolom belum lengkap: {required - set(cleaned.columns)}'
assert cleaned['label'].isin(['0', '1']).all(), 'Ada label di luar 0/1.'
cleaned['label'] = cleaned['label'].astype(int)
cleaned['duplicate_count'] = cleaned['duplicate_count'].astype(int)
assert not cleaned['komentar_processed'].duplicated().any(), 'Data input masih mengandung duplikat.'
assert not cleaned['komentar_processed'].str.strip().eq('').any(), 'Ada teks kosong.'
assert not {'is_synthetic', 'synthetic_id'}.intersection(cleaned.columns), 'Gunakan data bersih asli, bukan hasil augmentasi sebelumnya.'
assert isinstance(TARGET_ROWS, int) and TARGET_ROWS >= len(cleaned), 'Target harus bilangan bulat dan minimal sebesar data asli.'
cleaning_summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
SOURCE_SHA256 = hashlib.sha256(INPUT_PATH.read_bytes()).hexdigest()
assert SOURCE_SHA256 == cleaning_summary['source_sha256'], 'Sumber telah berubah sejak cleaning.'
assert len(cleaned) == cleaning_summary['cleaned_rows'], 'Jumlah data berbeda dari ringkasan cleaning.'
CLEANED_SHA256 = hashlib.sha256(cleaned_path.read_bytes()).hexdigest()
print(f'Data asli: {len(cleaned):,}; target total: {TARGET_ROWS:,}; tambahan: {TARGET_ROWS-len(cleaned):,}')
display(cleaned[['komentar', 'komentar_processed', 'label']].head())
display(cleaned['label'].value_counts().rename_axis('label').to_frame('jumlah_asli'))

Data asli: 8,934; target total: 10,000; tambahan: 1,066


,komentar,komentar_processed,label
0,Makin yakin abis baca review lain tentang ✌✌𝐒𝐆𝐈𝟖𝟖.,makin yakin abis baca review lain tentang ✌✌sgi88.,1
1,Paling suka model H2 😍🔥,paling suka model h2 😍🔥,0
2,Mobilnya udah hancur 🥺,mobilnya udah hancur 🥺,0
3,░𝙈𝘼𝙉𝙐𝙏88░benar2 bikin aku jadi sultan,░manut88░benar2 bikin aku jadi sultan,1
4,Semoga lekas recover mobilnya mas Dipo,semoga lekas recover mobilnya mas dipo,0


,jumlah_asli
label,
0,6016
1,2918


## 3. Fungsi cleaning yang konsisten

Aturan sama dengan notebook cleaning: NFKC → hapus karakter format Unicode → ubah kontrol menjadi spasi → huruf kecil → URL menjadi `<url>` → rapikan spasi. Angka, simbol, dan emoji dipertahankan. Generator memeriksa keunikan setelah transformasi ini.

In [3]:
url_pattern = re.compile(r'(?:https?://|www\.)[^\s<>]+', flags=re.IGNORECASE)

def clean_comment(text):
    text = unicodedata.normalize('NFKC', text)
    text = ''.join('' if unicodedata.category(c) == 'Cf' else c for c in text)
    text = ''.join(' ' if unicodedata.category(c) == 'Cc' else c for c in text)
    text = text.lower()
    text = url_pattern.sub('<url>', text)
    return re.sub(r'\s+', ' ', text).strip()

assert clean_comment('  𝐒𝐆𝐈𝟖𝟖  ') == 'sgi88'
assert clean_comment('cek HTTPS://contoh.test/a') == 'cek <url>'

## 4. Aturan dan generator komentar sintetis

Bagian ini menambahkan komentar **buatan berbasis template**, bukan komentar YouTube baru yang dikumpulkan. Jumlah yang ditambahkan dihitung dari hasil cleaning, bukan ditulis tetap. Semua tambahan diberi `label=1` sesuai tujuan membangun contoh promosi judi; pemetaan label tetap perlu dikonfirmasi dari sumber.

Gaya mengikuti ciri umum sampel positif: bahasa percakapan, testimoni, istilah slot/depo/WD/gacor, emoji, dan variasi huruf Unicode. Nama yang dipakai berupa penanda demo, tanpa tautan atau akun nyata. Isi testimoni dan klaim keuntungan adalah materi sintetis untuk deteksi, bukan fakta. Generator tidak menyalin komentar sumber secara utuh.

Dataset asli hasil cleaning tetap terpisah. `is_synthetic`, `data_origin`, `synthetic_id`, dan `generator_version` mencatat asal data. Metadata video, author, dan tanggal sintetis dibiarkan kosong, bukan direkayasa.

**Aturan eksperimen:** jangan membagi CSV gabungan secara acak langsung menjadi train/test. Bagi komentar asli terlebih dahulu dan gunakan tambahan sintetis hanya untuk data latih, dengan pengaturan ditetapkan tanpa melihat hasil data uji. Tuning/validasi juga memakai komentar asli yang terpisah. Bandingkan baseline tanpa augmentasi dengan eksperimen augmentasi pada data uji asli yang sama. Template dan nama demo bisa menjadi jalan pintas bagi model; keunikan teks tidak berarti ada 1.066 observasi independen. Laporkan jumlah asli dan sintetis secara terpisah di paper.

In [4]:
needed = TARGET_ROWS - len(cleaned)
if needed < 0:
    raise ValueError('Data asli melebihi target; notebook tidak akan membuang data asli untuk memenuhi target.')
rng = random.Random(SEED)

# Kerangka ditulis untuk augmentasi; tidak mengambil baris komentar sumber sebagai template.
openings = ['', 'bang, ', 'cuy ', 'jujur ya ', 'asli dah ', 'wkwk ', 'ga nyangka ', 'menurut gw ',
            'buat yg nanya, ', 'iseng doang awalnya, ', 'malem ini ', 'abis nonton ini ', 'eh serius ',
            'gw kira cuma rame doang, ', 'ternyata ', 'bro ', 'sumpah ', 'wah ', 'nah ini ', 'baru ngeh ']
templates = [
    'modal receh main slot, pas wd malah bikin senyum sendiri',
    'depo kecil dulu ternyata scatter nya nongol terus',
    'dari tadi cari pola slot akhirnya dapet yg gacor juga',
    'abis coba putaran receh malah dikasih maxwin',
    'slot malem ini pecah, modal tipis wd lumayan',
    'yg penting sabar nunggu scatter, abis itu wd deh',
    'baru daftar udah dapet bonus buat muter slot',
    'kirain slot biasa ternyata gampang banget dapet scatter',
    'depo dikit terus jackpot nyangkut, rejeki anak kos',
    'main slot sambil ngopi tau tau saldo naik',
    'putaran kecil aja udah dikasih bonus slot beruntun',
    'gacor juga ini slot, sampe lupa tadi modal berapa',
    'wd hasil slot hari ini lumayan buat jajan',
    'biasanya cuma liat testimoni slot, sekarang ikut wd',
    'baru beberapa spin slot langsung dikasih free spin',
    'niat coba bonus member baru malah betah muter slot',
    'adminnya cepet bales pas gw nanya depo buat slot',
    'slot abis magrib ini scatter nya royal amat',
    'pola spin receh gini ternyata bisa bikin slot pecah',
    'cuma modal saldo receh sekarang udah bisa tarik hasil slot',
    'semenjak nemu slot yg ini jadi paham kenapa orang bilang gacor',
    'td nyoba slot sebentar, ga nyangka wd nya cepet',
    'bonus deposit slot nya lumayan buat nambah putaran',
    'yg nyari slot gacor coba perhatiin jam mainnya juga',
    'akhirnya ngerasain jackpot slot setelah muter santai',
    'abis depo langsung cobain slot, scatter nya ga pelit',
    'slot ini free spin nya dateng pas saldo udah tipis',
    'nyoba bet kecil di slot dulu eh malah maxwin',
    'ga perlu modal gede buat ngerasain slot pecah hari ini',
    'wd slot pertama gw lancar, sampe sekarang masih kaget',
    'seneng banget slot receh gw akhirnya ngasih jackpot',
    'slot lagi rame bonus, depo receh dapet tambahan saldo',
    'baru pindah slot udah dapet scatter beruntun',
    'temen ngajak main slot receh, ternyata wd nya beneran masuk',
    'testi slot hari ini, depo tipis tariknya lumayan',
    'setelah nyobain bonus slot jadi ngerti istilah maxwin',
    'main slot ga buru buru, tau tau scatter turun',
    'modal jajan buat slot malah jadi saldo wd',
    'yang nanya slot nya gacor apa engga, gw udah wd tadi',
    'slot jam segini bikin betah, free spin nya sering mampir',
]
endings = ['', ' wkwk', ' mantul', ' lumayan buat ngopi', ' hoki hari ini', ' ga boong',
           ' auto senyum', ' semoga lanjut gacor', ' modal aman dulu', ' receh jadi lumayan',
           ' bikin nagih spin nya', ' seneng bgt', ' penasaran kan', ' gas tipis aja',
           ' yg penting wd dulu', ' rejeki ga kemana', ' enak juga ya', ' akhirnya cuan',
           ' makasih bonusnya', ' testi dikit nih']
emojis = ['', ' 🔥', ' 😂', ' 😎', ' 🙏', ' 💸', ' ✨', ' 🤣', ' 🔥🔥', ' 😭']
demo_sites = ['SITUSDEMO88', 'SLOTCONTOH77', 'DEMOSPIN99', 'CONTOHSLOT17']

def fancy_ascii(text):
    # Bold Unicode termasuk bentuk tulisan yang dijumpai pada spam sumber.
    return ''.join(chr(0x1D400 + ord(c) - ord('A')) if 'A' <= c <= 'Z'
                   else chr(0x1D41A + ord(c) - ord('a')) if 'a' <= c <= 'z'
                   else chr(0x1D7CE + ord(c) - ord('0')) if '0' <= c <= '9'
                   else c for c in text)

seen = set(cleaned['komentar_processed'])
synthetic_records = []
attempts = 0
while len(synthetic_records) < needed:
    attempts += 1
    if attempts > max(10_000, needed * 100):
        raise RuntimeError('Variasi template tidak cukup untuk menghasilkan teks unik sesuai target.')
    text = rng.choice(openings) + rng.choice(templates) + rng.choice(endings)
    if rng.random() < 0.35:
        site = rng.choice(demo_sites)
        text += ' di ' + (fancy_ascii(site) if rng.random() < 0.5 else site)
    style = rng.randrange(5)
    if style == 1:
        text = text.replace('slot', 'sl0t')
    elif style == 2:
        text = text.replace('slot', fancy_ascii('slot'))
    elif style == 3:
        text = text.replace('slot', 's l o t')
    elif style == 4:
        text = text.replace('depo', 'deposit').replace('gw', 'gua')
    text += rng.choice(emojis)
    processed = clean_comment(text)
    if not processed or processed in seen:
        continue
    seen.add(processed)
    record = {column: '' for column in cleaned.columns}
    record.update({
        'komentar': text, 'komentar_processed': processed, 'label': 1, 'duplicate_count': 1,
        'is_synthetic': True, 'data_origin': 'synthetic_template',
        'synthetic_id': f'SYN-{len(synthetic_records)+1:04d}',
        'generator_version': GENERATOR_VERSION,
    })
    synthetic_records.append(record)

original_for_augmentation = cleaned.copy()
original_for_augmentation['is_synthetic'] = False
original_for_augmentation['data_origin'] = 'source_dataset'
original_for_augmentation['synthetic_id'] = ''
original_for_augmentation['generator_version'] = ''
synthetic = pd.DataFrame(synthetic_records, columns=original_for_augmentation.columns)
augmented = pd.concat([original_for_augmentation, synthetic], ignore_index=True)
augmented['label'] = augmented['label'].astype(int)
augmented['is_synthetic'] = augmented['is_synthetic'].astype(bool)
print(f'Asli bersih: {len(cleaned):,}; sintetis: {len(synthetic):,}; total: {len(augmented):,}')
display(synthetic[['synthetic_id', 'komentar', 'komentar_processed', 'label']].head(12))
display(pd.crosstab(augmented['data_origin'], augmented['label'], margins=True))

Asli bersih: 8,934; sintetis: 1,066; total: 10,000


,synthetic_id,komentar,komentar_processed,label
0,SYN-0001,nah ini semenjak nemu slot yg ini jadi paham kenapa orang bilang gacor lumayan buat ngopi di 𝐃𝐄𝐌𝐎𝐒𝐏𝐈𝐍𝟗𝟗 💸,nah ini semenjak nemu slot yg ini jadi paham kenapa orang bilang gacor lumayan buat ngopi di demospin99 💸,1
1,SYN-0002,"asli dah modal receh main slot, pas wd malah bikin senyum sendiri akhirnya cuan di DEMOSPIN99 🔥","asli dah modal receh main slot, pas wd malah bikin senyum sendiri akhirnya cuan di demospin99 🔥",1
2,SYN-0003,eh serius adminnya cepet bales pas gw nanya depo buat sl0t semoga lanjut gacor ✨,eh serius adminnya cepet bales pas gw nanya depo buat sl0t semoga lanjut gacor ✨,1
3,SYN-0004,malem ini depo kecil dulu ternyata scatter nya nongol terus akhirnya cuan 🙏,malem ini depo kecil dulu ternyata scatter nya nongol terus akhirnya cuan 🙏,1
4,SYN-0005,"wkwk main 𝐬𝐥𝐨𝐭 ga buru buru, tau tau scatter turun lumayan buat ngopi 😭","wkwk main slot ga buru buru, tau tau scatter turun lumayan buat ngopi 😭",1
5,SYN-0006,"bang, modal jajan buat 𝐬𝐥𝐨𝐭 malah jadi saldo wd penasaran kan di DEMOSPIN99 🤣","bang, modal jajan buat slot malah jadi saldo wd penasaran kan di demospin99 🤣",1
6,SYN-0007,nah ini sl0t abis magrib ini scatter nya royal amat akhirnya cuan di DEMOSPIN99 🤣,nah ini sl0t abis magrib ini scatter nya royal amat akhirnya cuan di demospin99 🤣,1
7,SYN-0008,"bro s l o t malem ini pecah, modal tipis wd lumayan semoga lanjut gacor","bro s l o t malem ini pecah, modal tipis wd lumayan semoga lanjut gacor",1
8,SYN-0009,abis nonton ini cuma modal saldo receh sekarang udah bisa tarik hasil slot makasih bonusnya 😭,abis nonton ini cuma modal saldo receh sekarang udah bisa tarik hasil slot makasih bonusnya 😭,1
9,SYN-0010,menurut gw akhirnya ngerasain jackpot sl0t setelah muter santai modal aman dulu di SITUSDEMO88 😭,menurut gw akhirnya ngerasain jackpot sl0t setelah muter santai modal aman dulu di situsdemo88 😭,1


label,0,1,All
data_origin,,,
source_dataset,6016,2918,8934
synthetic_template,0,1066,1066
All,6016,3984,10000


## 5. Tinjau sebelum ekspor

Sel sebelumnya hanya membuat variabel `synthetic` dan `augmented` di memori. Periksa contoh mentah dan hasil cleaning sebelum menjalankan sel ekspor. Ubah template jika terlalu berulang; jika template diubah, ubah juga `GENERATOR_VERSION`. Bila seed atau template diubah, jalankan kembali generator sebelum ekspor.

Keunikan string tidak menjamin variasi makna. Nama demo dan pola template dapat menjadi petunjuk buatan yang terlalu mudah untuk model. Pisahkan data asli menjadi train/validasi/test terlebih dahulu pada notebook modeling; tambahan sintetis hanya untuk train. Jangan gunakan `is_synthetic`, ID, atau metadata sebagai fitur model. Gunakan `komentar_processed` sebagai input dan `label` sebagai target.

**Sel berikut menulis CSV** ke `outputs/augmentation/` dan menimpa hasil ekspor sebelumnya dengan nama sama. File asli hasil cleaning tetap utuh.

In [5]:
assert len(augmented) == TARGET_ROWS
assert len(synthetic) == needed
assert augmented['is_synthetic'].sum() == needed
assert not augmented['komentar_processed'].duplicated().any()
assert not augmented['komentar_processed'].str.strip().eq('').any()
assert augmented['label'].isin([0, 1]).all()
assert synthetic['label'].eq(1).all()
assert synthetic['komentar_processed'].map(clean_comment).equals(synthetic['komentar_processed'])
assert set(cleaned['komentar_processed']).isdisjoint(set(synthetic['komentar_processed']))
pd.testing.assert_frame_equal(augmented.iloc[:len(cleaned)][cleaned.columns].reset_index(drop=True), cleaned, check_dtype=False)
for metadata_column in ['source_row', 'video_id', 'title', 'channel_name', 'tanggal', 'author']:
    if metadata_column in synthetic:
        assert synthetic[metadata_column].eq('').all()
assert hashlib.sha256(INPUT_PATH.read_bytes()).hexdigest() == SOURCE_SHA256

assert hashlib.sha256(cleaned_path.read_bytes()).hexdigest() == CLEANED_SHA256, 'Input cleaning berubah saat notebook dijalankan.'
AUGMENT_DIR.mkdir(parents=True, exist_ok=True)
augmented_path = AUGMENT_DIR / f'komentar_{TARGET_ROWS}_with_synthetic.csv'
synthetic_path = AUGMENT_DIR / 'komentar_synthetic_only.csv'
augmented.to_csv(augmented_path, index=False, encoding='utf-8-sig')
synthetic.to_csv(synthetic_path, index=False, encoding='utf-8-sig')
augmentation_summary = {
    'target_rows': TARGET_ROWS, 'original_clean_rows': len(cleaned),
    'synthetic_rows': len(synthetic), 'total_rows': len(augmented),
    'label_counts': {str(k): int(v) for k, v in augmented['label'].value_counts().items()},
    'seed': SEED, 'generator_version': GENERATOR_VERSION,
    'method': 'manually authored templates with deterministic random combinations and style variations',
    'source_sha256': SOURCE_SHA256,
    'cleaned_csv_sha256': hashlib.sha256(cleaned_path.read_bytes()).hexdigest(),
    'augmented_csv_sha256': hashlib.sha256(augmented_path.read_bytes()).hexdigest(),
    'synthetic_label': 1,
    'label_mapping_pending_source_confirmation': True,
    'evaluation_policy': 'Split original data first. Synthetic examples for training only; validation and test contain original data only. Compare to baseline without augmentation.',
    'limitations': ['synthetic examples are not independent observations', 'template and demo-name shortcuts may inflate performance', 'source label quality is not established'],
}
(AUGMENT_DIR / 'augmentation_summary.json').write_text(json.dumps(augmentation_summary, ensure_ascii=False, indent=2), encoding='utf-8')
export_check = pd.read_csv(augmented_path, keep_default_na=False, dtype=str, encoding='utf-8-sig')
assert len(export_check) == TARGET_ROWS
assert export_check['is_synthetic'].eq('True').sum() == needed
assert export_check['komentar_processed'].tolist() == augmented['komentar_processed'].tolist()
print(f'Verifikasi lulus: tepat {TARGET_ROWS:,} komentar dengan asal data tercatat.')
for path in [augmented_path, synthetic_path, AUGMENT_DIR / 'augmentation_summary.json']:
    print(path.relative_to(PROJECT_DIR))

Verifikasi lulus: tepat 10,000 komentar dengan asal data tercatat.
outputs\augmentation\komentar_10000_with_synthetic.csv
outputs\augmentation\komentar_synthetic_only.csv
outputs\augmentation\augmentation_summary.json
